# Healthcare RAG vs. LLM — Batch Experiments

This notebook reproduces the comparative evaluation described in *"Evaluating RAG and LLM Architectures for Evidence-Grounded Healthcare AI"*.

**Pipeline:**
1. Load the healthcare QA dataset (`data/healthcare_dataset.csv`)
2. Ingest sample medical PDFs into a FAISS vector store (RAG knowledge base)
3. Run every question through both the baseline LLM and the RAG system
4. Score each response across all 8 evaluation metrics from Table I of the paper
5. Aggregate results, save `results/metrics.csv`, and generate the comparison report + charts

> **Prerequisite:** A running inference backend (default: Ollama with `llama3:8b-instruct` pulled). See the README's "Installation" section if you haven't set this up yet.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable when running from notebooks/
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import config
from llm_baseline import BaselineLLM, LLMBackendError
from rag_pipeline import RAGPipeline
from evaluation import (
    evaluate_llm_response,
    evaluate_rag_response,
    save_metrics_csv,
    results_to_dataframe,
    aggregate_by_system,
    generate_comparison_report,
    generate_evaluation_report,
)
from visualize import generate_all_charts

pd.set_option("display.max_colwidth", 80)

## 1. Load the QA dataset

In [ ]:
dataset = pd.read_csv(config.HEALTHCARE_DATASET_CSV)
print(f"Loaded {len(dataset)} questions across categories: {dataset['category'].unique().tolist()}")
dataset.head()

## 2. Ingest medical PDFs into the RAG vector store

Place PDFs in `data/sample_medical_papers/` first (see the README in that folder). This cell will skip gracefully if none are found, in which case you can still run the baseline LLM experiments below.

In [ ]:
rag = RAGPipeline()
pdf_paths = sorted(str(p) for p in config.SAMPLE_PAPERS_DIR.glob("*.pdf"))

if pdf_paths:
    print(f"Ingesting {len(pdf_paths)} PDF(s)...")
    rag.ingest_pdfs(pdf_paths)
    print("Vector store ready.")
else:
    print("No PDFs found in data/sample_medical_papers/. Add PDFs and re-run this cell before the RAG loop below.")

## 3. Run the baseline LLM over the full dataset

In [ ]:
llm = BaselineLLM()
llm_eval_results = []

for _, row in dataset.iterrows():
    try:
        response = llm.answer(row["question"])
        eval_result = evaluate_llm_response(response, reference_answer=row["reference_answer"])
        llm_eval_results.append(eval_result)
        print(f"[LLM] {row['question'][:60]}... -> accuracy={eval_result.factual_accuracy:.2f}")
    except LLMBackendError as e:
        print(f"[LLM] Skipped (backend unavailable): {row['question'][:60]}... ({e})")
        break

## 4. Run the RAG system over the full dataset

In [ ]:
rag_eval_results = []

if not pdf_paths:
    print("Skipping RAG loop: no PDFs ingested. See step 2.")
else:
    for _, row in dataset.iterrows():
        try:
            relevant_sources = [s.strip() for s in str(row["relevant_sources"]).split(";")]
            response = rag.answer(row["question"])
            eval_result = evaluate_rag_response(
                response,
                reference_answer=row["reference_answer"],
                relevant_sources=relevant_sources,
            )
            rag_eval_results.append(eval_result)
            print(f"[RAG] {row['question'][:60]}... -> accuracy={eval_result.factual_accuracy:.2f}")
        except LLMBackendError as e:
            print(f"[RAG] Skipped (backend unavailable): {row['question'][:60]}... ({e})")
            break

## 5. Aggregate, save, and report

In [ ]:
all_results = llm_eval_results + rag_eval_results

if all_results:
    save_metrics_csv(all_results)
    full_df = pd.read_csv(config.METRICS_CSV)
    display(aggregate_by_system(full_df))

    generate_comparison_report(full_df)
    generate_evaluation_report(full_df)
    chart_paths = generate_all_charts()
    print(f"Generated {len(chart_paths)} chart(s) in results/plots/")
else:
    print("No results were generated (inference backend was unavailable for the entire run).")

## 6. Inspect individual results

In [ ]:
if all_results:
    results_to_dataframe(all_results)[["question", "system", "factual_accuracy", "hallucination_rate", "clinical_safety_score", "latency_seconds"]]